## Running Abil

In [ ]:
#paths:
from pathlib import Path
#handling data:
import pandas as pd
from yaml import load
from yaml import CLoader as Loader
from datetime import datetime
#abil functions:
from abil.tune import ModelTuner as tune
from abil.predict import ModelPredictor as predict
from abil.post import AbilPostProcessor as post


In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "environment.yml").exists():
            return path
    raise FileNotFoundError("Could not find project root containing environment.yml")


PROJECT_ROOT = find_project_root()
conffile = PROJECT_ROOT / "1-phase example" / "regressor.yml"

# Load model configuration
with conffile.open('r') as f:
    model_config = load(f, Loader=Loader)

model_config['root'] = str(PROJECT_ROOT) + '/'
model_config['local_root'] = str(PROJECT_ROOT) + '/'


In [ ]:
# Load training data
targets = pd.read_csv(PROJECT_ROOT / model_config['targets'])
d = pd.read_csv(PROJECT_ROOT / model_config['training'])

# Define target
target = targets['Target'][0]

# Define predictors based on YAML
predictors = model_config['predictors']
d = d.dropna(subset=predictors)

# Split training data into X_train and y
y = d[target]
X_train = d[predictors]

print("finished loading data")


In [ ]:
#setup model:
m = tune(X_train, y, model_config)

#run model:
m.train(model='rf', log='both')
m.train(model='knn', log='both')
m.train(model='xgb', log='both')


In [ ]:
# Load prediction data
X_predict = pd.read_csv(PROJECT_ROOT / model_config['prediction'])
X_predict.set_index(["time", "depth", "lat", "lon"], inplace=True)
X_predict = X_predict[predictors]

# Setup model
m = predict(X_train, y, X_predict, model_config)

# Predict model
m.make_prediction()


In [ ]:
target_names = targets['Target'].values
target_subset = target_names[:1] # subset for estimate_applicability and merge_obs
current_date = datetime.today().strftime('%Y-%m-%d')

def do_post(statistic):
    m = post(X_train, y, X_predict, model_config, statistic)
    
    if statistic == "mean":
        print('begin estimate_applicability')
        m.estimate_applicability(target_subset)

        print('begin merge_obs')
        m.merge_obs(current_date,target_subset)

    print('begin export_ds')
    m.export_ds(current_date)

    print('begin integration')
    magnitude_conversion = 1e-21
    molar_mass = 12.01
    integ = m.integration(m, magnitude_conversion=magnitude_conversion,molar_mass=molar_mass,rate=True)
    if statistic == "mean":
        integ.integrated_totals(target_names)
    else:
        integ.integrated_totals(target_subset)

    print('do_post for: ', statistic, ' complete')

In [ ]:
do_post(statistic="mean")
do_post(statistic="ci95_UL")
do_post(statistic="ci95_LL")